In [1]:
import json
import pandas as pd
import numpy as np
import math as m
from datetime import datetime
from uuid import uuid4

## Data Preparation


In [2]:
def unique_id(df):
    current_ts = datetime.now()
    unique_ids = df['ts_gps_destination'].apply(lambda x: 
                                               x.timestamp() + x.microsecond/1000000 
                                               if pd.notna(x) else (current_ts.timestamp() + current_ts.microsecond/1000000))
    df['id'] = unique_ids
    return df

In [3]:
unecessary_columns =[
  "precipIntensity_source",
  "precipProbability_source",
  "temperature_source",
  "apparentTemperature_source",
  "dewPoint_source",
  "humidity_source",
  "pressure_source",
  "windSpeed_source", 
  "cloudCover_source",
  "uvIndex_source",
  "visibility_source",
  "precipIntensity_destination",
  "precipProbability_destination",
  "temperature_destination",
  "apparentTemperature_destination",
  "dewPoint_destination",
  "humidity_destination",
  "pressure_destination",
  "windSpeed_destination",
  "cloudCover_destination",
  "uvIndex_destination",
  "visibility_destination",
  "Traffic_Street_Name_source",
  "Traffic_Distance_source",
  "Traffic_Street_Name_destination",
  "Traffic_Distance_destination",
  "Pos_in_Ref_Round_source",
  "Pos_in_Ref_Round_destination",
  "device_source",
  "device_destination",
  "area_source",
  "area_destination",
  "syncref_source",
  "syncref_destination",
  "Scenario"
]

# Example: Read a Parquet file into a pandas DataFrame
v2x_file_path = "testparaquet/sidelink_dataframe.parquet"
cellular_df = pd.read_parquet(v2x_file_path)
print(cellular_df.columns)
cellular_df = cellular_df.drop(columns=unecessary_columns)
cellular_df = unique_id(cellular_df)


Index(['Source', 'Destination', 'Scenario', 'time_epoch', 'SNR', 'RSRP',
       'RSSI', 'NOISE POWER', 'RX_GAIN', 'SubFrame_NUMBER', 'SubFrame_LENGHT',
       'Rx_power', 'MCS', 'Received Packets', 'ts_gps_source',
       'Latitude_source', 'Longitude_source', 'Altitude_source',
       'speed_kmh_source', 'COG_source', 'precipIntensity_source',
       'precipProbability_source', 'temperature_source',
       'apparentTemperature_source', 'dewPoint_source', 'humidity_source',
       'pressure_source', 'windSpeed_source', 'cloudCover_source',
       'uvIndex_source', 'visibility_source', 'Traffic_Jam_Factor_source',
       'Traffic_Street_Name_source', 'Traffic_Distance_source',
       'Pos_in_Ref_Round_source', 'device_source', 'area_source',
       'ts_gps_destination', 'Latitude_destination', 'Longitude_destination',
       'Altitude_destination', 'speed_kmh_destination', 'COG_destination',
       'precipIntensity_destination', 'precipProbability_destination',
       'temperature_desti

# Experiment 1: Finding PDR using Logic Based on Number of Error Packets by Number of Packet Delivery Ratio


## Calculation of Packet Delivery Ratio(PDR) from Packet Error Ratio(PER) 
The total number of messages anticipated within the statistical interval is represented by (maxCount - minCount + 1)



$$
\text{PER} = \frac{Number of Error Packets(E)}{Number of Packets Sent (S)}
$$

$$
E = S - Received Packets(R)
$$

$$
=>\text{PER} = \frac{S - R}{S}
$$

$$
=> S \cdot \text{PER} = S - R
$$

$$
R = S (1 - \text{PER})
$$

$$
S = \frac{R}{1 - \text{PER}}
$$

$$
\text{PDR} = \frac{Number of Received Packets(R)}{Number of Packets Sent(S)}
$$

$$
\text{PDR} = \frac{R}{\frac{R}{1 - \text{PER}}}
$$

$$
\text{PDR} = 1 - \text{PER}
$$

See Reference [1] below for more details.

In [4]:
# Reference [1] and Formula is 
cellular_df['Packets_sent'] = cellular_df['Received Packets']/(1-cellular_df['Packet_error_ratio'])
cellular_df['pdr'] =1 - cellular_df['Packet_error_ratio']


Formula to Calculate Channel Bandwidth based on reference [2]
$$
\text{Bandwidth(B)} = \text{Number of Sub\_channels} \times \text{MCS}
$$
Formula to Calculate Throughput
$$
Throughput(C) = B \times \log_2(1 + \text{SNR})
$$



In [5]:
cellular_df['channel_bandwidth'] = cellular_df['Sub_channels'] * cellular_df['MCS'] # Reference [2]
edge_case_snr = cellular_df['SNR'] + 1
edge_case_snr[edge_case_snr <= 0] = 1e-9
cellular_df['throughput'] = cellular_df['channel_bandwidth'] * np.log2(edge_case_snr) 
cellular_df

,Source,Destination,time_epoch,SNR,RSRP,RSSI,NOISE POWER,RX_GAIN,SubFrame_NUMBER,SubFrame_LENGHT,...,Traffic_Jam_Factor_destination,distance,Packet_transmission_rate_hz,Sub_channels,Packet_error_ratio,id,Packets_sent,pdr,channel_bandwidth,throughput
timestamp,,,,,,,,,,,,,,,,,,,,,
2021-06-22 09:51:48+02:00,2,4,1.624348e+09,14.144092,-71.160341,-41.737488,0.002013,44.0,426.941176,2.470588,...,2.73675,30.631466,20,2,0.15,1.624356e+09,20.0,0.85,16,62.730931
2021-06-22 09:51:48+02:00,4,2,1.624348e+09,15.227671,-68.752400,-46.970770,0.004633,45.0,463.300000,0.300000,...,3.08926,30.631466,20,2,0.00,1.624356e+09,20.0,1.00,16,64.326145
2021-06-22 09:51:49+02:00,4,2,1.624348e+09,15.273688,-67.572225,-45.746560,0.006282,45.0,102.500000,0.150000,...,2.73675,27.982724,20,2,0.00,1.624356e+09,20.0,1.00,16,64.391509
2021-06-22 09:51:49+02:00,2,4,1.624348e+09,14.295098,-70.777137,-46.170142,0.002737,44.0,100.526316,4.789474,...,3.03573,27.982724,20,2,0.05,1.624356e+09,20.0,0.95,16,62.959959
2021-06-22 09:51:50+02:00,4,2,1.624348e+09,15.494669,-69.356312,-47.530865,0.007427,45.0,200.294118,1.470588,...,2.73675,24.855905,20,2,0.15,1.624356e+09,20.0,0.85,16,64.702847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-06-23 17:07:00+02:00,2,4,1.624461e+09,12.810686,-77.330367,-49.631543,0.000524,40.0,493.761905,3.666667,...,0.84534,13.994021,50,10,0.58,1.624468e+09,50.0,0.42,120,454.525572
2021-06-23 17:07:00+02:00,2,3,1.624461e+09,7.966860,-74.786447,-47.084520,0.002939,46.0,491.200000,3.400000,...,0.84534,16.196722,50,10,0.70,1.624468e+09,50.0,0.30,120,379.752345
2021-06-23 17:07:00+02:00,4,2,1.624461e+09,13.532171,-78.470617,-50.735069,0.000349,40.0,504.914286,1.428571,...,0.84534,13.994021,50,10,0.30,1.624468e+09,50.0,0.70,120,463.341399


In [6]:
ignore_filters = ['Scenario','ts_gps_source','ts_gps_destination']
def preprocess_nan_values(dataFrame,filters = []):
    """Replace NaN in categorical columns with empty string and maintain proper dtypes"""
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            dataFrame[col] = dataFrame[col].fillna('Unknown').astype(str)
        if col not in filters:
            dataFrame[col] = dataFrame[col].astype(float)
    
    return dataFrame

def preprocess_category_values(dataFrame,filters = []):
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            dataFrame[col] = dataFrame[col].fillna('Unknown').astype('category')
        if col not in filters:
            dataFrame[col] = dataFrame[col].astype(float)
    
    return dataFrame

In [7]:
cellular_df_modified = preprocess_nan_values(cellular_df,ignore_filters)

In [8]:
newData= cellular_df_modified.sort_values("timestamp").reset_index(drop=True)


In [9]:
newData.to_csv("cellular_df.csv")
print(f"Dataset of size: {len(newData)} is added")

Dataset of size: 325868 is added


### References

[1] Y. Mingxi, Y. Rundong, L. Yanheng, G. Yuming, W. Jian and Y. Zhihan, "Traffic Statistics and Analysis of Transmitter in C-V2X Communication," 2021 IEEE 93rd Vehicular Technology Conference (VTC2021-Spring), Helsinki, Finland, 2021, pp. 1-5, doi: 10.1109/VTC2021-Spring51267.2021.9448635.
Abstract: In the communication of devices based on C-V2X, packet error rate (PER) is an important metric to measure the communication performance of a device. As for the packet loss phenomenon, we usually focus on why the receiver did not successfully receive the message, and rarely focus on whether the transmitter actually sent the message.We usually consider the messages sending situation recorded by the application layer as the messages that should be received by the receiver (packets that are known to be not sent by the application layer due to application layer congestion control, etc., are not in the scope of this paper). However, in the actual communication process, there are some discrepancies between the real packets sent from the bottom layer and the application layer's records. In the 2020 C-V2X Large-scale Pilot Demonstration, when we analyzed the results and calculated the received PER of the devices, we were confused whether some of the devices did not send all the packets successfully. Based on this confusion, we defined the concept of transmitter traffic to represent the actual packet sending situation of the device. We designed a method to calculate transmitter traffic by using the "large-scale" data available, and conducted statistics on the transmitter traffic of more than 40 terminal companies, more than 10 chip module companies, and more than 50 participating devices. We analyzed the statistical results, and analyzed the possible reasons for the unsuccessful transmitter traffic.
keywords: {Performance evaluation;Vehicular and wireless technologies;Transmitters;Error analysis;Conferences;Measurement uncertainty;Packet loss;Transmitter Traffic;C-V2X;Communication Performance;Packet Error Rate},
URL: https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=9448635&isnumber=9448629

[2] Ancans, A., Petersons, E., Jerjomins, R., Grabs, E., Ancans, G., & Ipatovs, A. (2022). Evaluation of Received Signal Power Level and Throughput Depending on Distance to Transmitter in Testbed for Automotive WLAN IEEE 802.11ac Communication Network. Latvian Journal of Physics and Technical Sciences, 59, 3 - 12.